In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'DB_Pedia'
name_model = 'gpt-4o'
mode = 'few'
seed = 1
part = 3

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post12/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post13/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/OPENAI_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
from openai import OpenAI

In [10]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [11]:
OPENAI_API_KEY = credentials["OPENAI_API_KEY"]

In [12]:
client = OpenAI(api_key=OPENAI_API_KEY)

# 3. Functions

In [13]:
def few_shot_prompt(text):

    intro = (
        "Classify the topic of the following text from the DBpedia Ontology dataset.\n"
        "Respond only with a single digit (0–13) according to the category:\n"
        "0 = Company\n"
        "1 = EducationalInstitution\n"
        "2 = Artist\n"
        "3 = Athlete\n"
        "4 = OfficeHolder\n"
        "5 = MeanOfTransportation\n"
        "6 = Building\n"
        "7 = NaturalPlace\n"
        "8 = Village\n"
        "9 = Animal\n"
        "10 = Plant\n"
        "11 = Album\n"
        "12 = Film\n"
        "13 = WrittenWork\n"
        "Return only the digit (no words, no punctuation).\n"
    )

    few_shots = (
        "\nHere are some examples:\n\n"

        "Example 1:\n"
        "Text: \"Angstrem Group (Russian: ОАО «Ангстрем» named after angstrom) is a group of Russian companies one of the largest manufacturers of integrated circuits in Eastern Europe.\"\n"
        "Label: 0\n\n"

        "Example 2:\n"
        "Text: \"Kirk Balk Community College is a state school in Barnsley South Yorkshire England. It is a technology specialist college.\"\n"
        "Label: 1\n\n"

        "Example 3:\n"
        "Text: \"Brian Robert Setzer (born April 10 1959) is an American guitarist singer and songwriter. He first found widespread success in the early 1980s with the rockabilly revival group Stray Cats and later with The Brian Setzer Orchestra.\"\n"
        "Label: 2\n\n"

        "Example 4:\n"
        "Text: \"Colin Henry Turkington (born 21 March 1982) is a Northern Irish auto racing driver and 2009 British Touring Car Champion.\"\n"
        "Label: 3\n\n"

        "Example 5:\n"
        "Text: \"María Antonieta de Bográn (born 13 July 1955) is the former 1st Vice President of Honduras. She was a candidate for Vice President in the 2009 Honduran elections and served as national chairperson of the National Party.\"\n"
        "Label: 4\n\n"

        "Example 6:\n"
        "Text: \"The South African Class 8E of 1983 is a South African electric locomotive used in shunting service.\"\n"
        "Label: 5\n\n"

        "Example 7:\n"
        "Text: \"The Lefferts Historic House located in Brooklyn's Prospect Park is a historic home and museum built circa 1783.\"\n"
        "Label: 6\n\n"

        "Example 8:\n"
        "Text: \"Østensjøvannet is a lake in Oslo, Norway, known for its birdlife and wildlife preserve status.\"\n"
        "Label: 7\n\n"

        "Example 9:\n"
        "Text: \"Malgammana is a village located in the Central Province of Sri Lanka.\"\n"
        "Label: 8\n\n"

        "Example 10:\n"
        "Text: \"The Tawny Coster (Acraea terpsicore) is a butterfly species belonging to the Nymphalidae family.\"\n"
        "Label: 9\n\n"

        "Example 11:\n"
        "Text: \"Cupressus abramsiana is a species of cypress endemic to the Santa Cruz Mountains in California.\"\n"
        "Label: 10\n\n"

        "Example 12:\n"
        "Text: \"Kim Appleby is the solo debut album by British singer Kim Appleby released in 1990.\"\n"
        "Label: 11\n\n"

        "Example 13:\n"
        "Text: \"Yitzhak Rabin: a Biography is a 2004 documentary film about the life of Israeli Prime Minister Yitzhak Rabin.\"\n"
        "Label: 12\n\n"

        "Example 14:\n"
        "Text: \"The Cameo Murders is a book by Barry Shortall detailing a 1949 murder case in Liverpool.\"\n"
        "Label: 13\n\n"
    )

    target = f'Text: "{text}"\nLabel:'
    return intro + few_shots + target


In [14]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            top_p=1.0,
            seed=seed,
            stream=True,
            stream_options={"include_usage": True}
        )

        for chunk in stream:
            if first_token_time is None and len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                first_token_time = time.perf_counter()

            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                output_text += chunk.choices[0].delta.content

            if chunk.usage is not None:
                usage = chunk.usage

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": (
                (first_token_time - start_time) * 1000
                if first_token_time else None
            ),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }

    except:

        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-'
        }

# 4. Load Dataset

In [15]:
df = pd.read_csv(path_open)

In [16]:
df.shape

(1400, 36)

In [17]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [18]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = few_shot_prompt(text)
  output = predict_label(text, prompt)

  pred_label.append(int(output['prediction']))
  pred_latency.append(output['latency_ms'])
  pred_ttft.append(output['ttft_ms'])
  pred_input_tokens.append(output['input_tokens'])
  pred_output_tokens.append(output['output_tokens'])

  if (i % 10) == 0:
    print(i)

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390


In [19]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
gpt-4o-few-seed-1-label,
0,125
1,114
6,113
12,110
3,105
5,100
13,98
8,98
11,97


In [21]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,gpt-4o-few-seed-1-latency
count,1400.000000
mean,511.881862
std,328.945858
min,305.033318
25%,431.983209
50%,461.088761
75%,509.768102
max,7132.596353


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,gpt-4o-few-seed-1-ttft
count,1400.000000
mean,509.545022
std,328.714060
min,304.542528
25%,429.467199
50%,458.150468
75%,506.867975
max,7132.126973


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,gpt-4o-few-seed-1-input-tokens
count,1400.000000
mean,757.322857
std,29.422049
min,701.000000
25%,732.000000
50%,758.000000
75%,781.000000
max,880.000000


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,gpt-4o-few-seed-1-output-tokens
count,1400.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


# 5. Save Dataset

In [25]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [26]:
end_notebook = time.time()

In [27]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 12m 2.48s
